In [ ]:
from abc import ABC, abstractmethod
import threading

# ── Jerarquía de Personajes ──────────────────────────────────────────
class Personaje(ABC):
    def __init__(self, nombre: str):
        self.nombre = nombre
        self.nivel  = 1
        self.salud  = self._salud_base()

    @abstractmethod
    def _salud_base(self) -> int: ...

    @abstractmethod
    def atacar(self) -> str: ...

    def __str__(self):
        return f"[{self.__class__.__name__}] {self.nombre} | Niv:{self.nivel} | HP:{self.salud}"

class Guerrero(Personaje):
    def _salud_base(self): return 150
    def atacar(self): return f"{self.nombre} golpea con espada ⚔️"

class Mago(Personaje):
    def _salud_base(self): return 80
    def atacar(self): return f"{self.nombre} lanza bola de fuego 🔥"

class Arquero(Personaje):
    def _salud_base(self): return 110
    def atacar(self): return f"{self.nombre} dispara flecha 🏹"

# TAREA 1: nuevo tipo "paladin"
class Paladin(Personaje):
    def _salud_base(self): return 130
    def atacar(self): return f"{self.nombre} bendice con escudo sagrado 🛡️"

# ── Singleton: ConfiguracionJuego ────────────────────────────────────
class ConfiguracionJuego:
    _instancia = None
    _lock       = threading.Lock()

    @classmethod
    def get_instancia(cls):
        if cls._instancia is None:
            with cls._lock:
                if cls._instancia is None:
                    cls._instancia = ConfiguracionJuego()
        return cls._instancia

    def __init__(self):
        self.dificultad = "normal"
        self.idioma     = "es"
        self.volumen    = 80

    def mostrar(self):
        print(f"Config → dif:{self.dificultad} | idioma:{self.idioma} | vol:{self.volumen}")

# ── Simple Factory ────────────────────────────────────────────────────
class FabricaPersonajes:
    _catalogo = {
        "guerrero": Guerrero,
        "mago":     Mago,
        "arquero":  Arquero,
        "paladin":  Paladin,   # TAREA 1: registrado en el catálogo
    }

    @staticmethod
    def crear(tipo: str, nombre: str) -> Personaje:
        clase = FabricaPersonajes._catalogo.get(tipo.lower())
        if clase is None:
            raise ValueError(f"Tipo desconocido: '{tipo}'. Usa: {list(FabricaPersonajes._catalogo)}")
        personaje = clase(nombre)
        print(f"[Factory] Creado: {personaje}")
        return personaje

# ── TAREA 2: GestorMisiones Singleton ────────────────────────────────
class GestorMisiones:
    _instancia = None

    def __init__(self):
        self._misiones = []

    @classmethod
    def get_instancia(cls):
        if cls._instancia is None:
            cls._instancia = GestorMisiones()
        return cls._instancia

    def agregar_mision(self, mision: str):
        self._misiones.append(mision)
        print(f"[Misión] Agregada: {mision}")

    def listar(self):
        print("\n📋 Misiones activas:")
        for i, m in enumerate(self._misiones, 1):
            print(f"  {i}. {m}")

# ── TAREA 3: equipo de 4, todos atacan ───────────────────────────────
print("=" * 50)
cfg = ConfiguracionJuego.get_instancia()
cfg.mostrar()
print("=" * 50)

equipo = [
    FabricaPersonajes.crear("guerrero", "Thorin"),
    FabricaPersonajes.crear("mago",     "Gandalf"),
    FabricaPersonajes.crear("arquero",  "Legolas"),
    FabricaPersonajes.crear("paladin",  "Arthas"),
]

print("\n⚔️  RONDA DE COMBATE:")
for p in equipo:
    print(f"  → {p.atacar()}")

# ── TAREA 4: dos instancias directas de ConfiguracionJuego ───────────
print("\n--- TAREA 4: Prueba de doble instancia ---")
cfg1 = ConfiguracionJuego.get_instancia()
cfg2 = ConfiguracionJuego.get_instancia()
print(f"cfg1 is cfg2: {cfg1 is cfg2}")   # True → misma instancia
# Python permite ConfiguracionJuego() directamente (problema)
# La solución es lanzar RuntimeError en __init__ si _instancia existe

# ── GestorMisiones en uso ─────────────────────────────────────────────
gm = GestorMisiones.get_instancia()
gm.agregar_mision("Derrotar al Dragón de Fuego")
gm.agregar_mision("Recuperar el Orbe Oscuro")
gm.listar()